# V3 Semantic Layer Demo

This notebook demonstrates the **Phase 8 V3 semantic API** of the Predictive Log Anomaly Engine.

It shows:
- Ingesting events via `POST /v3/ingest` and receiving semantic enrichment
- Fetching explanations via `GET /v3/alerts/{alert_id}/explanation`
- Inspecting model and semantic layer status via `GET /v3/models/info`
- Viewing V3 metrics from `GET /metrics`

**Prerequisites:**
```bash
# Start the API with semantic enabled:
SEMANTIC_ENABLED=true docker compose -f docker/docker-compose.yml up
# Or run locally:
SEMANTIC_ENABLED=true DEMO_MODE=true python -m uvicorn src.api.app:create_app --factory --port 8000
```

In [1]:
import json
import time

try:
    import requests
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', '-q'])
    import requests

BASE_URL = 'http://localhost:8000'

def pretty(data):
    print(json.dumps(data, indent=2, default=str))

print('Base URL:', BASE_URL)

Base URL: http://localhost:8000


## 1. Check V3 Models Info

In [ ]:
resp = requests.get(f'{BASE_URL}/v3/models/info')
resp.raise_for_status()
info = resp.json()
print('=== /v3/models/info ===')
pretty(info)

print()
print(f'Inference mode:        {info["inference_mode"]}')
print(f'Artifacts loaded:      {info["artifacts_loaded"]}')
print(f'Semantic enabled:      {info["semantic_enabled"]}')
print(f'Semantic model loaded: {info["semantic_model_loaded"]}')

## 2. Check Health (V3 Semantic Component)

In [2]:
resp = requests.get(f'{BASE_URL}/health')
health = resp.json()
print('=== /health ===')
pretty(health)

semantic_component = health.get('components', {}).get('semantic', {})
print()
print('Semantic component:', semantic_component)

=== /health ===
{
  "status": "healthy",
  "uptime_seconds": 160.7,
  "components": {
    "inference_engine": {
      "status": "ok",
      "artifacts_loaded": true
    },
    "alert_manager": {
      "status": "ok"
    },
    "alert_buffer": {
      "status": "ok",
      "size": 66
    },
    "semantic": {
      "enabled": false,
      "model_loaded": false,
      "model_name": "all-MiniLM-L6-v2"
    }
  }
}

Semantic component: {'enabled': False, 'model_loaded': False, 'model_name': 'all-MiniLM-L6-v2'}


## 3. Ingest Events via POST /v3/ingest

With `DEMO_MODE=true` the engine uses a synthetic score so alerts fire without real trained models.

In [3]:
# Ingest enough events to fill a window (stride=1 in demo mode)
fired_alert = None
for i in range(15):
    payload = {
        'service': 'hdfs',
        'token_id': (abs(hash(f'demo-{i}')) % 7833) + 2,
        'session_id': 'v3-demo-session',
        'timestamp': time.time(),
    }
    resp = requests.post(f'{BASE_URL}/v3/ingest', json=payload)
    body = resp.json()
    if body.get('alert'):
        fired_alert = body['alert']
        print(f'Alert fired on event {i+1}!')
        break

if fired_alert:
    print('\n=== Alert payload (V3 semantic fields) ===')
    for field in ('alert_id', 'severity', 'score', 'explanation',
                  'semantic_similarity', 'evidence_tokens', 'top_similar_events'):
        print(f'  {field}: {fired_alert.get(field)}')
else:
    print('No alert fired yet — try increasing DEMO_SCORE or reducing window size.')

Alert fired on event 10!

=== Alert payload (V3 semantic fields) ===
  alert_id: db333669-2317-47f1-87c6-c6e345c26650
  severity: critical
  score: 100.0
  explanation: None
  semantic_similarity: None
  evidence_tokens: None
  top_similar_events: None


## 4. Fetch Explanation for a Specific Alert

In [4]:
if fired_alert:
    alert_id = fired_alert['alert_id']
    resp = requests.get(f'{BASE_URL}/v3/alerts/{alert_id}/explanation')
    print(f'=== /v3/alerts/{alert_id}/explanation ===')
    pretty(resp.json())
else:
    print('No alert available — run cell 3 first.')

=== /v3/alerts/db333669-2317-47f1-87c6-c6e345c26650/explanation ===
{
  "alert_id": "db333669-2317-47f1-87c6-c6e345c26650",
  "semantic_enabled": false,
  "explanation": null,
  "evidence_tokens": null,
  "semantic_similarity": null,
  "top_similar_events": null
}


## 5. List All Alerts (with V3 fields)

In [5]:
resp = requests.get(f'{BASE_URL}/alerts')
data = resp.json()
print(f'Total alerts in buffer: {data["count"]}')

for alert in data['alerts']:
    print()
    print(f'  ID:                  {alert["alert_id"]}')
    print(f'  Severity:            {alert["severity"]}')
    print(f'  Score:               {alert["score"]:.3f}')
    print(f'  Explanation:         {alert.get("explanation")}')
    print(f'  Semantic similarity: {alert.get("semantic_similarity")}')

Total alerts in buffer: 67

  ID:                  88885922-e29b-43fc-ae5c-ab3614637795
  Severity:            critical
  Score:               100.000
  Explanation:         None
  Semantic similarity: None

  ID:                  cdc2cd10-4cd8-49b6-b86c-0dd99a473b10
  Severity:            critical
  Score:               100.000
  Explanation:         None
  Semantic similarity: None

  ID:                  d3832dcc-a6ad-452e-a9e4-25a7969bb26b
  Severity:            critical
  Score:               100.000
  Explanation:         None
  Semantic similarity: None

  ID:                  05683e42-6054-42b9-8113-0b91185fb325
  Severity:            critical
  Score:               100.000
  Explanation:         None
  Semantic similarity: None

  ID:                  60dacffd-7cda-4903-b507-a3f481b935f6
  Severity:            critical
  Score:               100.000
  Explanation:         None
  Semantic similarity: None

  ID:                  fc4063ba-513e-4356-a664-dd0734342f08
  Severity: 

## 6. V3 Prometheus Metrics

In [6]:
resp = requests.get(f'{BASE_URL}/metrics')
lines = resp.text.splitlines()
v3_lines = [l for l in lines if 'semantic' in l]

print('=== V3 Semantic Metrics ===')
for line in v3_lines:
    print(line)

=== V3 Semantic Metrics ===
# HELP semantic_enrichments_total Total alerts enriched by the V3 semantic layer
# TYPE semantic_enrichments_total counter
semantic_enrichments_total 0.0
# HELP semantic_enrichments_created Total alerts enriched by the V3 semantic layer
# TYPE semantic_enrichments_created gauge
semantic_enrichments_created 1.7753932998006542e+09
# HELP semantic_enrichment_latency_seconds Latency of V3 semantic enrichment per alert in seconds
# TYPE semantic_enrichment_latency_seconds histogram
semantic_enrichment_latency_seconds_bucket{le="0.001"} 0.0
semantic_enrichment_latency_seconds_bucket{le="0.005"} 0.0
semantic_enrichment_latency_seconds_bucket{le="0.01"} 0.0
semantic_enrichment_latency_seconds_bucket{le="0.025"} 0.0
semantic_enrichment_latency_seconds_bucket{le="0.05"} 0.0
semantic_enrichment_latency_seconds_bucket{le="0.1"} 0.0
semantic_enrichment_latency_seconds_bucket{le="0.25"} 0.0
semantic_enrichment_latency_seconds_bucket{le="0.5"} 0.0
semantic_enrichment_laten